# 03 Feature Engineering

Este notebook construye los datasets supervisados para predecir la **anomalía de precipitación costera a +3 meses**.

Se generan cuatro experimentos comparables:

- **A — ENSO:** índices ENSO + lags + estacionalidad del mes objetivo.
- **B — ENSO + precipitación reciente.**
- **C — B + variables atmosféricas locales de ERA5-Land.**
- **D — C + variables hidrológicas locales de ERA5-Land.**

La climatología se calcula **solo con el período de entrenamiento** para evitar *data leakage*.

In [38]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

HORIZON = 3
TRAIN_END = pd.Timestamp("2010-12-01")
VAL_END = pd.Timestamp("2017-12-01")

ENSO_PRECIP_FILE = DATA_PROCESSED / "enso_precip.csv"
ERA5_LOCAL_FILE = DATA_PROCESSED / "era5_local_features.csv"

print("Proyecto:", PROJECT_ROOT)
print("Horizonte:", HORIZON, "meses")

Proyecto: /home/gustavo-paredes/Documents/Developer/Yachay/enso-ml
Horizonte: 3 meses


## 1. Cargar y unir las fuentes

In [39]:
df = pd.read_csv(
    ENSO_PRECIP_FILE,
    parse_dates=["date"]
)

local_df = pd.read_csv(
    ERA5_LOCAL_FILE,
    parse_dates=["date"]
)

# Metadatos de ERA5 que no son predictores.
local_df = local_df.drop(
    columns=["number", "expver"],
    errors="ignore"
)

df = df.sort_values("date").reset_index(drop=True)
local_df = local_df.sort_values("date").reset_index(drop=True)

print("ENSO + precipitación:", df.shape)
print("ERA5 local:", local_df.shape)
print("Rango ERA5:", local_df["date"].min(), "->", local_df["date"].max())

ENSO + precipitación: (912, 12)
ERA5 local: (912, 13)
Rango ERA5: 1950-01-01 00:00:00 -> 2025-12-01 00:00:00


In [40]:
# Validaciones antes del merge
assert not df["date"].duplicated().any(), "Hay fechas duplicadas en enso_precip.csv"
assert not local_df["date"].duplicated().any(), "Hay fechas duplicadas en era5_local_features.csv"

df = pd.merge(
    df,
    local_df,
    on="date",
    how="left",
    validate="one_to_one"
)

print("Dataset integrado:", df.shape)
df.head()

Dataset integrado: (912, 24)


,date,nino12,nino3,nino34,nino4,soi,tni,mei,oni,precipitation_mm,...,u10_ms,v10_ms,wind_speed_ms,surface_pressure_hpa,evaporation_mm,runoff_mm,soil_water_1,soil_water_2,soil_water_3,soil_water_4
0,1950-01-01,-1.20,-1.34,-1.05,-0.69,0.54,0.624,NaN,-1.53,225.615542,...,0.612021,0.437981,0.934633,932.04193,73.816581,77.133245,0.366950,0.350620,0.327577,0.377396
1,1950-02-01,-1.27,-1.60,-1.50,-1.10,1.58,0.445,NaN,-1.34,295.464787,...,0.458641,0.205236,0.679810,932.30330,68.222015,127.091787,0.379683,0.376162,0.355740,0.382328
2,1950-03-01,-0.68,-0.96,-1.07,-0.91,1.79,0.382,NaN,-1.16,252.391600,...,0.550594,0.218168,0.734493,932.71936,90.915420,120.328326,0.387393,0.393051,0.388167,0.396222
3,1950-04-01,-1.22,-0.98,-0.91,-0.76,1.67,0.311,NaN,-1.18,307.984600,...,0.366579,0.398198,0.798530,932.15240,83.923831,185.992613,0.405306,0.401859,0.394797,0.402872
4,1950-05-01,-0.61,-1.33,-1.30,-0.79,0.55,0.124,NaN,-1.07,144.824903,...,0.487092,0.486566,0.953303,932.67975,80.675658,92.131619,0.349731,0.361851,0.368473,0.402260


## 2. Construir la anomalía sin leakage

La climatología mensual se calcula únicamente con observaciones cuya fecha es anterior o igual al final del período de entrenamiento (`2010-12-01`). Esa misma climatología se aplica luego a validación y prueba.

In [41]:
df["month"] = df["date"].dt.month

train_climatology = (
    df.loc[df["date"] <= TRAIN_END]
      .groupby("month")["precipitation_mm"]
      .mean()
)

assert len(train_climatology) == 12, "La climatología de entrenamiento no contiene los 12 meses."

df["climatology_mm"] = df["month"].map(train_climatology)

df["precip_anomaly"] = (
    df["precipitation_mm"]
    - df["climatology_mm"]
)

train_climatology

month
1     324.663879
2     370.387311
3     398.849610
4     346.349142
5     266.107109
6     187.664631
7     158.699567
8     137.969504
9     154.659881
10    171.603695
11    163.513859
12    232.210664
Name: precipitation_mm, dtype: float64

## 3. Crear target a +3 meses

In [42]:
# Verificamos que la serie temporal esté ordenada y no tenga meses faltantes.
expected_dates = pd.date_range(
    df["date"].min(),
    df["date"].max(),
    freq="MS"
)

assert len(expected_dates) == len(df), (
    "La serie no es mensual continua; no conviene usar shift() hasta revisar las fechas."
)
assert np.array_equal(expected_dates.values, df["date"].values), (
    "Las fechas no coinciden con una serie mensual continua."
)

TARGET_COLUMN = f"target_precip_anomaly_t{HORIZON}"

df["target_date"] = (
    df["date"] + pd.DateOffset(months=HORIZON)
)

df[TARGET_COLUMN] = (
    df["precip_anomaly"].shift(-HORIZON)
)

df[
    ["date", "precip_anomaly", "target_date", TARGET_COLUMN]
].head(8)

,date,precip_anomaly,target_date,target_precip_anomaly_t3
0,1950-01-01,-99.048337,1950-04-01,-38.364542
1,1950-02-01,-74.922525,1950-05-01,-121.282206
2,1950-03-01,-146.458010,1950-06-01,-28.006418
3,1950-04-01,-38.364542,1950-07-01,7.574480
4,1950-05-01,-121.282206,1950-08-01,6.524933
5,1950-06-01,-28.006418,1950-09-01,-8.525452
6,1950-07-01,7.574480,1950-10-01,-62.110724
7,1950-08-01,6.524933,1950-11-01,-83.619135


## 4. Estacionalidad del mes objetivo

Como el horizonte está fijado en +3 meses, codificamos el **mes que se desea predecir**, no el mes de origen.

In [43]:
df["target_month"] = df["target_date"].dt.month

df["target_month_sin"] = np.sin(
    2 * np.pi * df["target_month"] / 12
)

df["target_month_cos"] = np.cos(
    2 * np.pi * df["target_month"] / 12
)

## 5. Definir familias de variables

In [44]:
enso_variables = [
    "nino12",
    "nino3",
    "nino34",
    "nino4",
    "soi",
    "tni",
]

atmospheric_variables = [
    "temperature_c",
    "dewpoint_c",
    "u10_ms",
    "v10_ms",
    "surface_pressure_hpa",
]

hydrological_variables = [
    "evaporation_mm",
    "runoff_mm",
    "soil_water_1",
    "soil_water_2",
    "soil_water_3",
    "soil_water_4",
]

required_local = atmospheric_variables + hydrological_variables
missing_local = [c for c in required_local if c not in df.columns]

assert not missing_local, f"Faltan variables ERA5: {missing_local}"

df[required_local].isna().sum()

temperature_c           0
dewpoint_c              0
u10_ms                  0
v10_ms                  0
surface_pressure_hpa    0
evaporation_mm          0
runoff_mm               0
soil_water_1            0
soil_water_2            0
soil_water_3            0
soil_water_4            0
dtype: int64

## 6. Crear memoria temporal

Para cada predictor dinámico usamos el valor actual `t` y los lags `t-1`, `t-2` y `t-3`.  
La precipitación reciente también se incorpora como valor actual y tres lags.

In [45]:
LAGS = [1, 2, 3]

def add_lags(dataframe, variables, lags=LAGS):
    for variable in variables:
        for lag in lags:
            dataframe[f"{variable}_lag{lag}"] = dataframe[variable].shift(lag)

# ENSO
add_lags(df, enso_variables)

# Precipitación local reciente
add_lags(df, ["precip_anomaly"])

# Atmósfera local
add_lags(df, atmospheric_variables)

# Hidrología local
add_lags(df, hydrological_variables)

## 7. Construir las listas de features A, B, C y D

In [46]:
def current_and_lags(variables, lags=LAGS):
    columns = []
    for variable in variables:
        columns.append(variable)
        columns.extend(
            [f"{variable}_lag{lag}" for lag in lags]
        )
    return columns

enso_feature_columns = current_and_lags(enso_variables)

seasonal_features = [
    "target_month_sin",
    "target_month_cos",
]

precip_features = current_and_lags(
    ["precip_anomaly"]
)

atmospheric_features = current_and_lags(
    atmospheric_variables
)

hydrological_features = current_and_lags(
    hydrological_variables
)

# A: ENSO
feature_columns_a = (
    enso_feature_columns
    + seasonal_features
)

# B: ENSO + precipitación reciente
feature_columns_b = (
    feature_columns_a
    + precip_features
)

# C: B + atmósfera local
feature_columns_c = (
    feature_columns_b
    + atmospheric_features
)

# D: C + hidrología local
feature_columns_d = (
    feature_columns_c
    + hydrological_features
)

print("A — ENSO:", len(feature_columns_a))
print("B — + precipitación:", len(feature_columns_b))
print("C — + atmósfera:", len(feature_columns_c))
print("D — + hidrología:", len(feature_columns_d))

A — ENSO: 26
B — + precipitación: 30
C — + atmósfera: 50
D — + hidrología: 74


In [47]:
# Conteos esperados
assert len(feature_columns_a) == 26
assert len(feature_columns_b) == 30
assert len(feature_columns_c) == 50
assert len(feature_columns_d) == 74

# No debe entrar metadata ni target dentro de X.
forbidden = {
    "date",
    "target_date",
    TARGET_COLUMN,
    "climatology_mm",
    "target_month",
}

for name, columns in {
    "A": feature_columns_a,
    "B": feature_columns_b,
    "C": feature_columns_c,
    "D": feature_columns_d,
}.items():
    assert not (forbidden & set(columns)), (
        f"El experimento {name} contiene columnas prohibidas: "
        f"{forbidden & set(columns)}"
    )
    assert len(columns) == len(set(columns)), (
        f"El experimento {name} contiene features duplicadas."
    )

print("Validación de listas de features: OK")

Validación de listas de features: OK


## 8. Construir datasets supervisados

In [48]:
METADATA_COLUMNS = [
    "date",
    "target_date",
    "precip_anomaly",
]

def build_model_dataset(dataframe, feature_columns):
    columns = (
        METADATA_COLUMNS
        + feature_columns
        + [TARGET_COLUMN]
    )

    return (
        dataframe[columns]
        .dropna()
        .reset_index(drop=True)
    )

model_df_a = build_model_dataset(df, feature_columns_a)
model_df_b = build_model_dataset(df, feature_columns_b)
model_df_c = build_model_dataset(df, feature_columns_c)
model_df_d = build_model_dataset(df, feature_columns_d)

datasets = {
    "A": model_df_a,
    "B": model_df_b,
    "C": model_df_c,
    "D": model_df_d,
}

for name, data in datasets.items():
    print(
        name,
        "| shape:", data.shape,
        "| date:", data["date"].min().date(),
        "->", data["date"].max().date(),
        "| target:", data["target_date"].min().date(),
        "->", data["target_date"].max().date(),
    )

A | shape: (886, 30) | date: 1950-04-01 -> 2025-09-01 | target: 1950-07-01 -> 2025-12-01
B | shape: (886, 34) | date: 1950-04-01 -> 2025-09-01 | target: 1950-07-01 -> 2025-12-01
C | shape: (886, 54) | date: 1950-04-01 -> 2025-09-01 | target: 1950-07-01 -> 2025-12-01
D | shape: (886, 78) | date: 1950-04-01 -> 2025-09-01 | target: 1950-07-01 -> 2025-12-01


### Chequeo de comparabilidad

Para comparar A/B/C/D limpiamente, verificamos que validación y prueba cubran las mismas fechas objetivo. Si los datasets tienen el mismo rango temporal (como debería ocurrir con ERA5 completo y sin faltantes), este chequeo pasará.

In [49]:
for name, data in datasets.items():
    print(
        name,
        "Train:", (data["target_date"] <= TRAIN_END).sum(),
        "| Validation:",
        data["target_date"].between(
            TRAIN_END + pd.offsets.MonthBegin(1),
            VAL_END
        ).sum(),
        "| Test:", (data["target_date"] > VAL_END).sum()
    )

A Train: 710 | Validation: 84 | Test: 92
B Train: 710 | Validation: 84 | Test: 92
C Train: 710 | Validation: 84 | Test: 92
D Train: 710 | Validation: 84 | Test: 92


## 9. Guardar datasets A–D

In [50]:
output_files = {
    "A": DATA_PROCESSED / "enso_precip_model_t3_a.csv",
    "B": DATA_PROCESSED / "enso_precip_model_t3_b.csv",
    "C": DATA_PROCESSED / "enso_precip_model_t3_c.csv",
    "D": DATA_PROCESSED / "enso_precip_model_t3_d.csv",
}

for name, data in datasets.items():
    data.to_csv(
        output_files[name],
        index=False
    )
    print(f"{name}: {output_files[name].name} -> {data.shape}")

A: enso_precip_model_t3_a.csv -> (886, 30)
B: enso_precip_model_t3_b.csv -> (886, 34)
C: enso_precip_model_t3_c.csv -> (886, 54)
D: enso_precip_model_t3_d.csv -> (886, 78)


## 10. Resumen

El notebook deja cuatro datasets con el mismo target y el mismo horizonte.  
La única diferencia entre A, B, C y D es la información disponible para los modelos, lo cual permite atribuir cualquier cambio en las métricas a la incorporación progresiva de nuevas familias de variables.